# Riemann–Roch spaces, explicitly — plane curves, and a birational map from a cubic surface to $\mathbb{P}^2$

Written against the MSE question *"How to compute the Riemann–Roch space of a divisor on an algebraic
variety?"*, which has no answers. Part I does curves in detail; Part II does the case that motivated
the question: a smooth cubic surface carrying a Galois-stable set of three skew lines.

## 0. The one mechanism

Everything below is the same trick. Let $X \subset \mathbb{P}^n$ be projective, $D$ an **explicit
divisor** on $X$ (not merely a class), and $h$ any form of degree $m$ with $\operatorname{div}_X(h) - D \ge 0$.
Multiplication by $h$ is an isomorphism
$$L(D) \;\xrightarrow{\ \sim\ }\; \big\{\varphi \text{ of degree } m \bmod I_X \;:\;
\operatorname{div}_X(\varphi) \ \ge\ \operatorname{div}_X(h) - D\big\}, \qquad f \longmapsto fh .$$
The right-hand side is a system of **linear conditions on the coefficients of $\varphi$**. So the
algorithm is: choose a denominator, impose vanishing, do linear algebra, divide back.

Two things have to be got right, and they are exactly where the difficulty lives:

1. **$D$ must be an actual divisor.** $L$ of a divisor *class* is only defined up to multiplication by
   a function, so a class alone is not enough input. This will bite hard in Part II.
2. **The vanishing conditions must have the right multiplicities.** On a smooth $X$ with $D$ supported
   on smooth subvarieties this is just "vanish to order $\ge n_i$ along $Z_i$" and is linear. On a
   *singular plane model* of a curve it is **not** — that is the whole content of the classical
   adjoint/conductor theory (§I.3).

A remark on the sheaf-theoretic phrasing in the question: once $D$ is an honest divisor there is
nothing to choose. $\mathcal{O}_X(D)$ is *defined* by $\mathcal{O}_X(D)(U) = \{f : \operatorname{div} f + D|_U \ge 0\}$,
so $H^0(X,\mathcal{O}_X(D)) = L(D)$ on the nose. (With the convention $\operatorname{div} f + D \ge 0$ the sheaf is
$\mathcal{O}(D)$, not $\mathcal{O}(-D)$.)

# Part I — Curves

## I.1 Smooth plane curves: the mechanism is literally linear algebra

For $C \subset \mathbb{P}^2$ smooth of degree $d$, every divisor of interest can be written
$D = mH - A$ with $A$ effective and cut out by explicit equations, $H$ the hyperplane class. Taking
$h$ of degree $m$,
$$L(mH - A) \;=\; \tfrac{1}{h}\,\{\varphi \text{ of degree } m : \varphi|_A = 0\} .$$

In [1]:
R.<x,y,z> = QQ[]
d = 4
C = x^4 + y^4 + z^4 - 3*x^2*y*z                  # a smooth plane quartic
Jac = R.ideal([C] + [C.derivative(v) for v in (x,y,z)])
print("smooth ?", Jac.dimension() == 0, "   genus =", (d-1)*(d-2)//2)

def forms(m):
    return [x^i*y^j*z^(m-i-j) for i in range(m+1) for j in range(m+1-i)]

def L_of_mH_minus_A(m, pts):
    "basis of { forms of degree m vanishing at the points of A }, as a matrix kernel"
    if not pts:
        return identity_matrix(QQ, len(forms(m))).rows()
    M = Matrix(QQ, [[f(*P) for f in forms(m)] for P in pts])
    return M.right_kernel().basis()

print()
print("  D            deg   dim L(D)   deg+1-g   h^1")
g = (d-1)*(d-2)//2
for m, pts, name in [(1, [], "H"), (2, [], "2H"), (3, [], "3H")]:
    dim = len(L_of_mH_minus_A(m, pts)); deg = m*d
    print("  " + name.ljust(12) + str(deg).rjust(4) + str(dim).rjust(10)
          + str(deg+1-g).rjust(10) + str(dim-(deg+1-g)).rjust(6))

smooth ? True    genus = 3

  D            deg   dim L(D)   deg+1-g   h^1
  H              4         3         2     1
  2H             8         6         6     0
  3H            12        10        10     0


$\dim L(H) = 3$ while $\deg H + 1 - g = 2$: the discrepancy is $h^1 = h^0(K-H) = h^0(\mathcal{O}) = 1$,
since for a smooth plane quartic $K = (d-3)H = H$. So **$L(K)$ = the linear forms**, of dimension
$3 = g$ — the canonical system of a smooth plane curve is cut by the forms of degree $d-3$, which is
the cleanest instance of the mechanism.

Now a divisor that is not a multiple of $H$: take two rational points on $C$ and compute $L(2H - P - Q)$.

In [2]:
P2 = ProjectiveSpace(QQ, 2)
Cv = Curve(P2.coordinate_ring()(C), P2)
pts = [tuple(p) for p in Cv.rational_points(bound=4)][:2]
print("two rational points on C:", pts)
B = L_of_mH_minus_A(2, pts)
print("dim L(2H - P - Q) =", len(B), "   deg = 8-2 = 6,   deg+1-g =", 6+1-3)
print("a basis, as conics (to be divided by any fixed conic h):")
for v in B:
    print("   ", sum(c*f for c, f in zip(v, forms(2))))

two rational points on C: [(-1, 1, 1), (1, 1, 1)]
dim L(2H - P - Q) = 4    deg = 8-2 = 6,   deg+1-g = 4
a basis, as conics (to be divided by any fixed conic h):
    -x^2 + z^2
    -x^2 + y*z
    -x^2 + y^2
    -x*y + x*z


### Cross-check against the general machinery

Sage (like Magma) implements Hess's algorithm — ideal arithmetic in orders of the function field —
which needs no plane model and no adjoint bookkeeping. It wants a global function field, so we check
over $\mathbb{F}_p$.

In [3]:
k = GF(101)
P2k = ProjectiveSpace(k, 2)
Ck = Curve(P2k.coordinate_ring()(C), P2k)
print("genus over F_101:", Ck.genus())
pl = Ck.places(1)
D = 3*pl[0] + 2*pl[1]
print("D = 3*P1 + 2*P2, deg", D.degree(),
      "   dim L(D) =", len(D.basis_function_space()),
      "   deg+1-g =", D.degree()+1-Ck.genus())
print("first basis element:", D.basis_function_space()[0])

genus over F_101: 3


D = 3*P1 + 2*P2, deg 5    dim L(D) = 3    deg+1-g = 3
first basis element: 1


## I.2 What goes wrong for a singular plane model

Take the quartic
$$Q:\quad x^2y^2 + y^2z^2 + z^2x^2 - 2xyz(x+y+z) = 0 ,$$
which has three nodes and therefore **geometric genus $0$**, while its plane model has arithmetic
genus $3$. If we run the recipe of §I.1 blindly we compute on the *singular* model and get the wrong
answer.

In [4]:
Qf = x^2*y^2 + y^2*z^2 + z^2*x^2 - 2*x*y*z*(x+y+z)
Isng = R.ideal([Qf] + [Qf.derivative(v) for v in (x,y,z)])
nodes = [tuple(1 if i == j else 0 for i in range(3)) for j in range(3)]
print("singular locus:", [list(pp.gens()) for pp in Isng.radical().primary_decomposition()])
print("nodes:", nodes)
print()
print("arithmetic genus of the plane model:", (4-1)*(4-2)//2)
print("geometric genus (3 nodes)           :", (4-1)*(4-2)//2 - 3)
print()
print("naive 'L(H)' = linear forms                      : dim", len(forms(1)))
print("the truth: H pulls back to a degree-4 divisor on")
print("the normalisation P^1, so h^0 = 4 + 1 - 0        : dim", 5)

singular locus: [[z, y], [y, x], [z, x]]
nodes: [(1, 0, 0), (0, 1, 0), (0, 0, 1)]

arithmetic genus of the plane model: 3
geometric genus (3 nodes)           : 0

naive 'L(H)' = linear forms                      : dim 3
the truth: H pulls back to a degree-4 divisor on
the normalisation P^1, so h^0 = 4 + 1 - 0        : dim 5


So the naive count is $3$ and the truth is $5$. The point is that a rational function on the curve
may have a pole at a node "on one branch only", and no form on the plane sees that. The classical fix
is the **conductor**: for a node $\delta = 1$ and the conductor contributes $2\delta = 2$ to a divisor
$\Delta$ on the normalisation, so here $\deg\Delta = 6$.

**Brill–Noether.** Call $\varphi$ of degree $m$ an *adjoint* if $\operatorname{div}\varphi \ge \Delta$ — for
ordinary nodes this just means "passes through the nodes". An adjoint of degree $m$ cuts
$\operatorname{div}\varphi = \Delta + E_\varphi$ with $\deg E_\varphi = md - \deg\Delta$. Given $D$, pick an
adjoint $\psi$ of degree $m$ with $E_\psi \ge D_-$ and set $R = E_\psi - D$; then
$$L(D) \;=\; \tfrac{1}{\psi}\,\{\varphi \text{ adjoint of degree } m : E_\varphi \ge R\}.$$
For $D = $ the pullback of a line (degree $4$) we need $\deg E_\psi = 4m - 6 \ge 4$, so $m = 3$:
cubic adjoints form a $10 - 3 = 7$-dimensional space, $\deg R = 6 - 4 = 2$, and we get
$7 - 2 = 5$ — the right answer.

In [5]:
print("cubic adjoints (cubics through the 3 nodes):",
      len(L_of_mH_minus_A(3, nodes)), "= 10 - 3")
print("deg E_psi = 4*3 - 6 =", 4*3-6, "   deg R = 6 - 4 =", 2)
print("dim L(H*) = 7 - 2 =", 7-2, "   matches h^0(P^1, O(4)) =", 5)
print()
k2 = GF(101)
P2k2 = ProjectiveSpace(k2,2); Qk = Curve(P2k2.coordinate_ring()(Qf), P2k2)
print("Sage's function field, over F_101:  genus =", Qk.genus())
plq = Qk.places(1)
Dq = 4*plq[0]
print("  deg 4 divisor: dim L(D) =", len(Dq.basis_function_space()),
      "  (= 4+1-0, as it must be on a rational curve)")

cubic adjoints (cubics through the 3 nodes): 7 = 10 - 3
deg E_psi = 4*3 - 6 = 6    deg R = 6 - 4 = 2
dim L(H*) = 7 - 2 = 5    matches h^0(P^1, O(4)) = 5

Sage's function field, over F_101:  genus = 0


  deg 4 divisor: dim L(D) = 5   (= 4+1-0, as it must be on a rational curve)


## I.3 The payoff: parametrising the nodal quartic

Adjoints do more than fix dimension counts — they *produce the maps*. The conics through the three
nodes form a $3$-dimensional adjoint system cutting $\deg E = 2\cdot4 - 6 = 2$, i.e. a $g^2_2$ on the
normalisation: the map they define is the degree-$2$ Veronese $\mathbb{P}^1 \to \mathbb{P}^2$, so $Q$ is
carried birationally onto a **conic**. The conics through the three coordinate nodes are spanned by
$xy, yz, zx$ — the classical Cremona involution.

In [6]:
S.<u,v,w> = QQ[]
T.<x1,y1,z1,u1,v1,w1> = QQ[]
J = T.ideal([T(Qf.subs({x:x1, y:y1, z:z1})), u1 - x1*y1, v1 - y1*z1, w1 - z1*x1])
E = J.elimination_ideal([x1, y1, z1])
img = [S(gg.subs({u1:u, v1:v, w1:w})) for gg in E.gens()]
print("image of Q under (xy : yz : zx):", img)
con = Conic(img[0])
print("is a smooth conic ?", con.is_smooth(), "   has a rational point ?", con.has_rational_point())
par = con.parametrization()[0]
print("parametrisation of the conic:", par)

image of Q under (xy : yz : zx): [u^2 - 2*u*v + v^2 - 2*u*w - 2*v*w + w^2]
is a smooth conic ? True    has a rational point ? True
parametrisation of the conic: Scheme morphism:
  From: Projective Space of dimension 1 over Rational Field
  To:   Projective Conic Curve over Rational Field defined by u^2 - 2*u*v + v^2 - 2*u*w - 2*v*w + w^2
  Defn: Defined on coordinates by sending (x : y) to
        (x^2 - 2*x*y + y^2 : x^2 + 2*x*y + y^2 : 4*y^2)


In [7]:
# pull the parametrisation back to Q:  P^1 -> conic -> Q,  inverting the Cremona (xy:yz:zx)
P1.<s,t> = QQ[]
uu, vv, ww = [P1(gg) for gg in par.defining_polynomials()]
# Cremona is an involution: (u:v:w) -> (uw : uv : vw)  recovers (x:y:z)
cand = [uu*ww, uu*vv, vv*ww]
gg = gcd(gcd(cand[0], cand[1]), cand[2])
cand = [c//gg for c in cand]
print("candidate parametrisation of Q:", cand)
print("lies on Q ?", Qf(cand[0], cand[1], cand[2]) == 0)
print("degree:", max(c.degree() for c in cand), " (a nodal quartic is parametrised by quartics)")

candidate parametrisation of Q: [4*s^2*t^2 - 8*s*t^3 + 4*t^4, s^4 - 2*s^2*t^2 + t^4, 4*s^2*t^2 + 8*s*t^3 + 4*t^4]
lies on Q ? True
degree: 4  (a nodal quartic is parametrised by quartics)


That is a complete, machine-checked parametrisation of a singular quartic obtained purely from a
Riemann–Roch computation on the plane model. Note the two separate ingredients: the *adjoint system*
gave the map to a conic (pure linear algebra), and a *rational point on the conic* was needed to
finish. That split — linear algebra, then an arithmetic obstruction — is exactly the shape of Part II.

# Part II — A cubic surface with three skew conjugate lines

## II.1 The example

Blow up $\mathbb{P}^2$ at six points forming two Galois orbits of size three,
$$\{(1:\alpha:\alpha^2) : \alpha^3 = 2\} \quad\text{and}\quad \{(\beta^2:1:\beta) : \beta^3 = 3\},$$
and embed by the cubics through them. The conditions are Galois-stable, so everything is computed over
$\mathbb{Q}$ by linear algebra: a cubic $F$ vanishes on the first orbit iff $F(1,\alpha,\alpha^2) = 0$ in
$\mathbb{Q}[\alpha]/(\alpha^3-2)$, which is three $\mathbb{Q}$-linear conditions.

In [8]:
U.<T> = QQ[]
A.<al> = NumberField(T^3 - 2); Bf.<be> = NumberField(T^3 - 3)
RA = R.change_ring(A); RB = R.change_ring(Bf)

def orbit_rows(mons, pt, K, RK):
    vals = [K(RK(m)(*pt)) for m in mons]
    return [[QQ(vv.list()[i]) for vv in vals] for i in range(K.degree())]

ptA, ptB = (A(1), al, al^2), (be^2, Bf(1), be)
M3 = Matrix(QQ, orbit_rows(forms(3), ptA, A, RA) + orbit_rows(forms(3), ptB, Bf, RB))
print("cubics through the six points:", M3.right_kernel().dimension())
M2c = Matrix(QQ, orbit_rows(forms(2), ptA, A, RA) + orbit_rows(forms(2), ptB, Bf, RB))
print("conics through the six points:", M2c.right_kernel().dimension(), " (general position)")

f = [2*x^3 - 11*x*y*z + 5*z^3, -5*x*y^2 + x^2*z + 2*y*z^2,
     4*x^2*y + 3*y^2*z - 5*x*z^2, 4*x^3 + 15*y^3 - 17*x*y*z]
print("basis of the cubic system (scaled):")
for gg in f: print("   ", gg)

cubics through the six points: 4
conics through the six points: 0  (general position)
basis of the cubic system (scaled):
    2*x^3 - 11*x*y*z + 5*z^3
    -5*x*y^2 + x^2*z + 2*y*z^2
    4*x^2*y + 3*y^2*z - 5*x*z^2
    4*x^3 + 15*y^3 - 17*x*y*z


In [9]:
# the cubic relation among the f_i : the surface
S4.<Y0,Y1,Y2,Y3> = QQ[]; Yv = [Y0,Y1,Y2,Y3]
from itertools import combinations_with_replacement as cwr
idx  = list(cwr(range(4), 3))
mons9 = forms(9)
rows = [[ (f[t[0]]*f[t[1]]*f[t[2]]).monomial_coefficient(m) for m in mons9] for t in idx]
rel  = Matrix(QQ, rows).left_kernel().basis()
print("cubic relations among f0..f3:", len(rel))
Fs = sum(c*prod(Yv[i] for i in t) for c, t in zip(rel[0], idx))
Fs = Fs*lcm(c.denominator() for c in Fs.coefficients())
print("S :", Fs, "= 0")
Ising = S4.ideal([Fs] + [Fs.derivative(vv) for vv in Yv])
print("smooth ?", Ising.dimension() - 1 == -1)

cubic relations among f0..f3: 1
S : -30*Y1^3 - 17*Y0*Y1*Y2 + 5*Y2^3 + 2*Y0^2*Y3 + 11*Y1*Y2*Y3 - Y0*Y3^2 = 0
smooth ? True


## II.2 The three skew lines

The exceptional curve over $(1:\alpha:\alpha^2)$ maps to a line $L_\alpha \subset S$. The planes
containing it are exactly the members of the cubic system that are **singular** at the point, i.e. the
kernel of $c \mapsto \sum_i c_i\,\nabla f_i(P_\alpha)$ — again linear algebra, now over
$F = \mathbb{Q}(\alpha)$.

In [10]:
SA.<Z0,Z1,Z2,Z3> = A[]; Zv = [Z0,Z1,Z2,Z3]
FA = SA(Fs)
Jm = Matrix(A, [[A(RA(gg).derivative(RA(vv))(*ptA)) for vv in (x,y,z)] for gg in f])
ker = Jm.left_kernel().basis()
h = [sum(vv[i]*Zv[i] for i in range(4)) for vv in ker]
print("L_alpha = V(h1, h2) with")
print("   h1 =", h[0]); print("   h2 =", h[1])
print("L_alpha lies on S ?", FA.reduce(SA.ideal(h).groebner_basis()) == 0)

L_alpha = V(h1, h2) with
   h1 = Z0 + (23/8*al^2)*Z2 - 5/4*Z3
   h2 = Z1 + (5/4*al)*Z2 + (-1/4*al^2)*Z3


L_alpha lies on S ?

 True


In [11]:
# the unique quadric through the three conjugate lines  (they are pairwise skew iff it is smooth)
Pst.<s2,t2> = A[]
param = [-23/8*al^2*s2 + 5/4*t2, -5/4*al*s2 + 1/4*al^2*t2, s2, t2]
M2f = forms(2)
rows = []
for m in [S4(mm.subs({x:Y0, y:Y1, z:Y2})) for mm in []] or [Yv[i]*Yv[j] for i,j in cwr(range(4),2)]:
    val = SA(m)(*param)
    co = [val.monomial_coefficient(s2^2), val.monomial_coefficient(s2*t2), val.monomial_coefficient(t2^2)]
    rows.append([c for cc in co for c in A(cc).list()])
Mq = Matrix(QQ, rows).transpose()
print("quadrics through the three conjugate lines:", Mq.right_kernel().dimension())
qv = Mq.right_kernel().basis()[0]
Quad = sum(c*m for c, m in zip(qv, [Yv[i]*Yv[j] for i,j in cwr(range(4),2)]))
Quad = Quad*lcm(c.denominator() for c in Quad.coefficients())
print("Q :", Quad)
G = Matrix(QQ, 4, 4, lambda i, j: Quad.derivative(Yv[i]).derivative(Yv[j]))
print("Gram determinant:", G.det(), "  => smooth, so the three lines are pairwise skew")

quadrics through the three conjugate lines: 1
Q : 40*Y0^2 + 529*Y1*Y2 - 54*Y0*Y3 + 5*Y3^2
Gram determinant: 592143556   => smooth, so the three lines are pairwise skew


Three pairwise skew lines lie on a unique quadric, and it is defined over $\mathbb{Q}$ because the set of
lines is. This is the first structural fact: the Galois-stable triple is genuinely "spread out" on $S$.

## II.3 Step one: contracting the three lines is free

Projection from $L_\alpha$ is just $\pi_\alpha = (h_1 : h_2)$, and $|H - L_\alpha|$ is the corresponding
conic-bundle class. Over $\bar{\mathbb{Q}}$ the product $(\pi_1,\pi_2,\pi_3)$ contracts exactly
$L_1,L_2,L_3$ and is birational onto a del Pezzo surface of degree $6$, cut out by a $(1,1,1)$-form in
$(\mathbb{P}^1)^3$. Over $\mathbb{Q}$ this is a single map into a Weil restriction,
$$S \dashrightarrow R_{F/\mathbb{Q}}\,\mathbb{P}^1_F, \qquad
\mathbf{Y} \longmapsto \big(h_1(\mathbf{Y}) : h_2(\mathbf{Y})\big) \in \mathbb{P}^1(F),$$
and it costs nothing but the two linear forms. **This step is easy and completely explicit — the
difficulty in the problem is entirely in the second step.**

## II.4 Step two: the class, and what it takes to write it down

We want a Galois-invariant class $D$ with $D^2 = 1$ and $D\cdot K = -3$; then $h^0(D) = 3$ and
$\phi_{|D|}$ is the map. Working in $\operatorname{Pic}(\bar S) = \langle \ell, E_1,\dots,E_6\rangle$:

In [12]:
Mlat = diagonal_matrix(ZZ, [1,-1,-1,-1,-1,-1,-1])
def dot(a, b): return vector(ZZ,a)*Mlat*vector(ZZ,b)
lc = (1,0,0,0,0,0,0)
Ee = [tuple([0]+[1 if j==i else 0 for j in range(6)]) for i in range(6)]
Kc = tuple(vector(ZZ,(-3,0,0,0,0,0,0)) + sum(vector(ZZ,e) for e in Ee)); Hc = tuple(-vector(ZZ,Kc))
lines = list(Ee)
lines += [tuple(vector(ZZ,lc)-vector(ZZ,Ee[i])-vector(ZZ,Ee[j])) for i in range(6) for j in range(i+1,6)]
lines += [tuple(2*vector(ZZ,lc)-sum(vector(ZZ,Ee[j]) for j in range(6) if j!=i)) for i in range(6)]
print("27 lines ?", len(lines) == 27)
Tri = [Ee[0], Ee[1], Ee[2]]
skew = [Lq for Lq in lines if all(dot(Lq,t)==0 for t in Tri) and Lq not in Tri]
from itertools import combinations
tri2 = [c for c in combinations(skew,3) if all(dot(a,b)==0 for a,b in combinations(c,2))]
print("lines skew to all three:", len(skew), "   pairwise-skew triples among them:", len(tri2))
Lvee = tuple(2*vector(ZZ,lc) - sum(vector(ZZ,Ee[j]) for j in range(1,6)))
print("L^vee (meets E2..E6, skew to E1) is a line:", Lvee in lines)
print("H + E1 - L^vee == l ?", tuple(vector(ZZ,Hc)+vector(ZZ,Ee[0])-vector(ZZ,Lvee)) == lc)
D2 = tuple(2*vector(ZZ,Hc) - vector(ZZ,lc))
print("2H - l : square", dot(D2,D2), " .K", dot(D2,Kc), " degree", dot(D2,Hc),
      "  chi =", 1+(dot(D2,D2)-dot(D2,Kc))//2)

27 lines ? True
lines skew to all three: 6    pairwise-skew triples among them: 2
L^vee (meets E2..E6, skew to E1) is a line: True
H + E1 - L^vee == l ? True
2H - l : square 1  .K -3  degree 3   chi = 3


Three facts come out of that:

* the lines skew to all of $L_1,L_2,L_3$ number exactly **six**, splitting into exactly **two**
  pairwise-skew triples — the two triples of the hexagon on the del Pezzo surface of degree 6. If
  Galois preserves them (rather than swapping them) we get a Galois-stable **sixer**, hence an
  invariant $\ell$;
* $\ell = H + E - L^\vee$, where $E$ is one line of the sixer and $L^\vee$ the unique line meeting the
  other five and skew to $E$;
* $2H - \ell$ has square $1$, $\cdot K = -3$, **degree $3$** and $h^0 = 3$: its members are twisted
  cubics on $S$ (possibly degenerating to conic $+$ line).

Combining, and using $h^0(\mathcal{O}_S(2H)) = 10$:
$$\boxed{\;L(\ell) \;=\; \tfrac{1}{q_\Gamma}\{\text{quadrics of } \mathbb{P}^3 \text{ containing } \Gamma\},
\qquad \Gamma \in |2H-\ell| \text{ a twisted cubic on } S. \;}$$
A twisted cubic imposes $h^0(\mathcal{O}_\Gamma(2)) = 7$ conditions on the $10$ quadrics, leaving a **net** —
exactly the $3$ sections we want. This is the concrete form of the map, and it is again pure linear
algebra *once $\Gamma$ is known*. Everything now turns on producing $\Gamma$, and that is where the
field of definition enters.

## II.5 Over $F = \mathbb{Q}(\alpha)$: the degenerate $\Gamma$ is free

$C' + L^\vee$ is a degenerate member of $|2H-\ell|$: take a plane through $E = L_\alpha$, let $C'$ be
the residual conic, and take $L^\vee$ as above. Both are computed from the three lines alone.

In [13]:
# L^vee : the image of the conic through the five points other than P_alpha
UA.<tt> = A[]
quadfac = tt^2 + al*tt + al^2                     # the other two roots of T^3-2, over F
def alpha_pair_rows(mons):
    polys = [UA(RA(m)(A(1), tt, tt^2)) % quadfac for m in mons]
    return [[p.list()[i] if len(p.list()) > i else A(0) for p in polys] for i in range(2)]
Mc5 = Matrix(A, [[A(c) for c in r] for r in orbit_rows(forms(2), ptB, Bf, RB)] + alpha_pair_rows(forms(2)))
C5 = sum(cc*RA(m) for cc, m in zip(Mc5.right_kernel().basis()[0], forms(2)))
print("conic through the other five points:", C5)
xa, ya, za = RA.gens()
cols = [RA(gg) for gg in f] + [-C5*xa, -C5*ya, -C5*za]
Kv = Matrix(A, [[gg.monomial_coefficient(RA(m)) for gg in cols] for m in forms(3)]).right_kernel().basis()
gl = [sum(vv[i]*Zv[i] for i in range(4)) for vv in Kv]
print("L^vee = V(g1, g2):"); print("   g1 =", gl[0]); print("   g2 =", gl[1])
print("on S ?", FA.reduce(SA.ideal(gl).groebner_basis()) == 0,
      "   skew to L_alpha ?",
      Matrix(A, [[p.monomial_coefficient(vv) for vv in Zv] for p in [h[0], h[1], gl[0], gl[1]]]).det() != 0)

conic through the other five points: (-1/5*al)*x^2 - x*y + (3/4*al^2)*y^2 + (-1/4*al^2)*x*z + (3/5*al)*y*z + z^2


L^vee = V(g1, g2):
   g1 = Z0 + (13/20*al^2)*Z2 - 3/10*Z3
   g2 = Z1 + (2/5*al)*Z2 + (1/10*al^2)*Z3
on S ? True    skew to L_alpha ? True


In [14]:
# residual conic C' of the plane h1, by an ideal quotient; then the net of quadrics
Ic  = (SA.ideal([FA, h[0]])).quotient(SA.ideal(h))
Jnet = Ic.intersection(SA.ideal(gl))
Mon2 = [Zv[i]*Zv[j] for i, j in cwr(range(4), 2)]
Gb = Jnet.groebner_basis()
reds = [m.reduce(Gb) for m in Mon2]
allm = sorted(set().union(*[set(r.monomials()) for r in reds]))
Ker = Matrix(A, [[r.monomial_coefficient(mm) for mm in allm] for r in reds]).left_kernel().basis()
qsF = [sum(c*m for c, m in zip(vv, Mon2)) for vv in Ker]
print("C' ideal:", Ic.gens())
print("dim of quadrics through C' + L^vee:", len(qsF), "  (predicted 3)")
for q in qsF: print("   ", q)

C' ideal: [8*Z0 + (23*al^2)*Z2 - 10*Z3, 120*Z1^2 + (-150*al)*Z1*Z2 + (-8*al^2)*Z2^2 + (30*al^2)*Z1*Z3 - 109*Z2*Z3 + (15*al)*Z3^2]
dim of quadrics through C' + L^vee: 3   (predicted 3)
    Z0^2 + (423/32*al^2)*Z1^2 - 2115/64*Z1*Z2 + (-2927/160*al)*Z2^2 + 35/32*Z0*Z3 + (423/64*al)*Z1*Z3 + (-67/40*al^2)*Z2*Z3 + 3/8*Z3^2
    Z0*Z1 + (3/2*al)*Z1^2 + (al^2)*Z1*Z2 - 1/5*Z2^2 + (1/4*al^2)*Z0*Z3 - 1/2*Z1*Z3 + (3/40*al)*Z2*Z3 + (-1/8*al^2)*Z3^2
    -15/4*Z1^2 + Z0*Z2 + (75/16*al)*Z1*Z2 + (25/8*al^2)*Z2^2 + (-3/8*al)*Z0*Z3 + (-15/16*al^2)*Z1*Z3


In [15]:
# verification: compose with the blow-up map; the result must be a linear automorphism of P^2
pb = [q(f[0], f[1], f[2], f[3]) for q in qsF]
gcd3 = gcd(gcd(pb[0], pb[1]), pb[2])
lin = [p//gcd3 for p in pb]
print("pullbacks have degree", [p.degree() for p in pb], " with common factor of degree", gcd3.degree())
print("cofactors:", lin)
Mlin = Matrix(A, [[l.monomial_coefficient(vv) for vv in RA.gens()] for l in lin])
print("all linear ?", all(l.degree() == 1 for l in lin), "   invertible ?", Mlin.det() != 0)
print()
print("the degree-5 common factor factors as (conic)*(cubic), i.e. C5 and the pullback of C':")
print(factor(gcd3))

pullbacks have degree [6, 6, 6]  with common factor of degree 5
cofactors: [(25/128*al^2)*x + (-3/80*al)*y + 1/16*z, 1/40*y, -1/16*x]
all linear ? True    invertible ? True

the degree-5 common factor factors as (conic)*(cubic), i.e. C5 and the pullback of C':
((48*al)) * (x^2 + (5/2*al^2)*x*y + (-15/4*al)*y^2 + (5/4*al)*x*z - 3*y*z + (-5/2*al^2)*z^2) * (x^3 + (-23/6*al^2)*x^2*y + 25/4*y^3 - 41/12*x*y*z + (-23/8*al^2)*y^2*z + (115/24*al^2)*x*z^2 - 5/3*z^3)


So over $F$ the map is written down, and it is birational: composing with the blow-up gives a linear
automorphism of $\mathbb{P}^2$. **This much follows from the three skew lines alone.**

## II.6 Over $\mathbb{Q}$: the descent

The obvious idea — change basis by $M^{-1}$ to make the map rational — fails, and it is worth seeing
why: the identification $L(\ell) \cong \{\text{quadrics}\}$ is by multiplication by the section cutting
$\Gamma$, and *that section is irrational*. Two nets built from different $\Gamma$'s differ by
multiplication by a ratio of quadrics, not by a constant matrix.

In [16]:
Mnew = Mlin.inverse()*vector(SA, qsF)
print("after the change of basis M^-1, are the quadrics rational ?")
for i, q in enumerate(Mnew):
    q = q*lcm(c.denominator() for c in q.coefficients())
    print("   q%d rational ? %s" % (i, all(c in QQ for c in q.coefficients())))
print()
print("=> what is needed is a Q-RATIONAL Gamma in |2H - l|, not a change of basis.")

after the change of basis M^-1, are the quadrics rational ?
   q0 rational ? False
   q1 rational ? False
   q2 rational ? False

=> what is needed is a Q-RATIONAL Gamma in |2H - l|, not a change of basis.


What one actually needs is a $\mathbb{Q}$-rational twisted cubic $\Gamma \in |2H-\ell|$. The class is
invariant, so $|2H-\ell|$ is a Severi–Brauer surface over $\mathbb{Q}$; a rational $\Gamma$ exists exactly
when it splits, which it does as soon as $S(\mathbb{Q}) \neq \emptyset$. Finding it is the descent step —
and §II.7 explains why that step is not merely laborious but is the whole arithmetic content.

In this example we can produce one, because $2H - \ell$ pulls back to the quintics with double points
at the six blown-up points — and *those conditions are Galois-stable*, hence solvable over $\mathbb{Q}$.

In [17]:
def double_point_rows(mons, pt, K, RK):
    rows = []
    for vv in R.gens():
        vals = [K(RK(m).derivative(RK(vv))(*pt)) for m in mons]
        for i in range(K.degree()):
            rows.append([QQ(val.list()[i]) for val in vals])
    return rows
Mq5 = Matrix(QQ, double_point_rows(forms(5), ptA, A, RA) + double_point_rows(forms(5), ptB, Bf, RB))
print("quintics with double points at the six points:", Mq5.right_kernel().dimension())
sq = sum(c*m for c, m in zip(Mq5.right_kernel().basis()[0], forms(5)))
sq = sq*lcm(c.denominator() for c in sq.coefficients())
print("s =", sq)

quintics with double points at the six points: 3
s = -32*x^4*y + 375*x*y^4 + 2*x^2*y^2*z + 55*x^3*z^2 - 168*y^3*z^2 - 240*x*y*z^3 + 100*z^5


In [18]:
# the quadrics whose pullback is divisible by s : a Q-rational net
cols = [m(f[0], f[1], f[2], f[3]) for m in [Yv[i]*Yv[j] for i,j in cwr(range(4),2)]] + [-sq*vv for vv in R.gens()]
KQ = Matrix(QQ, [[gg.monomial_coefficient(m) for gg in cols] for m in forms(6)]).right_kernel().basis()
qsQ = [sum(c*m for c, m in zip(vv[:10], [Yv[i]*Yv[j] for i,j in cwr(range(4),2)])) for vv in KQ]
print("dimension of the rational net:", len(qsQ))
for q in qsQ:
    q = q*lcm(c.denominator() for c in q.coefficients())
    print("   ", q)

dimension of the rational net: 3
    4*Y0^2 - 3*Y1*Y2 - 2*Y0*Y3
    10*Y0*Y1 - 2*Y2^2 - 5*Y1*Y3
    -15*Y1^2 + 4*Y0*Y2


In [19]:
pbQ = [q(f[0], f[1], f[2], f[3]) for q in qsQ]
gq = gcd(gcd(pbQ[0], pbQ[1]), pbQ[2])
linQ = [p//gq for p in pbQ]
MQ = Matrix(QQ, [[l.monomial_coefficient(vv) for vv in (x,y,z)] for l in linQ])
print("common factor degree:", gq.degree(), "   cofactors:", linQ)
print("linear and invertible ?", all(l.degree()==1 for l in linQ), MQ.det() != 0)
print()
print("So, over Q, the birational map S --> P^2 is")
print("   (Y0:Y1:Y2:Y3)  |-->  (4Y0^2 - 3Y1Y2 - 2Y0Y3  :  10Y0Y1 - 2Y2^2 - 5Y1Y3  :  4Y0Y2 - 15Y1^2)")
chk = [4*Y0^2-3*Y1*Y2-2*Y0*Y3, 10*Y0*Y1-2*Y2^2-5*Y1*Y3, 4*Y0*Y2-15*Y1^2]
print("matches the computed net ?",
      Matrix(QQ,[[c.monomial_coefficient(m) for m in [Yv[i]*Yv[j] for i,j in cwr(range(4),2)]]
                 for c in chk]).row_space()
      == Matrix(QQ,[[c.monomial_coefficient(m) for m in [Yv[i]*Yv[j] for i,j in cwr(range(4),2)]]
                 for c in qsQ]).row_space())

common factor degree: 5    cofactors: [-1/4*z, -1/10*y, 1/4*x]
linear and invertible ? True True

So, over Q, the birational map S --> P^2 is
   (Y0:Y1:Y2:Y3)  |-->  (4Y0^2 - 3Y1Y2 - 2Y0Y3  :  10Y0Y1 - 2Y2^2 - 5Y1Y3  :  4Y0Y2 - 15Y1^2)
matches the computed net ? True


## II.7 The Severi–Brauer surface is the point, not a wart of the method

It is tempting to read §II.6 as saying that the descent is extra labour. It is not. There exist smooth
cubic surfaces with a Galois-stable triple of skew lines and **no rational points at all**, and for
those there is no birational map to $\mathbb{P}^2$ to be written down, by any method: such an $S$ is
birational to a non-trivial Severi–Brauer surface, hence not $k$-rational.

**They can be built to order.** Let $P/k$ be a non-trivial Severi–Brauer surface. It has closed points
of degree $3$; blow one up in general position to get a del Pezzo surface $Y$ of degree $6$, then blow
up a degree-$3$ point of $Y$ in general position to get a smooth cubic surface $S$. The last three
exceptional curves are a Galois-stable triple of skew lines, and $S(k) = \emptyset$ by Nishimura's
lemma, since $S$ is birational to $P$.

**There are two Severi–Brauer surfaces, and they are opposite.** Contracting the sixer
$\{L_1,L_2,L_3\}\cup T$ gives one; contracting $\{L_1,L_2,L_3\}\cup T'$ gives another. Writing $\ell_1,
\ell_2$ for the two blow-down classes and $M = L_1+L_2+L_3$, the lattice identity is
$$\ell_1 + \ell_2 \;=\; H + M .$$
Now $\delta : \operatorname{Pic}(\bar S)^{G} \to \operatorname{Br}(k)$, the connecting map of
$0 \to \operatorname{Pic}(S) \to \operatorname{Pic}(\bar S)^{G} \to \operatorname{Br}(k)$, is a **homomorphism**, and it kills
$H = -K_S$ and $M$ (a Galois-stable sum of lines, hence a rational divisor). Therefore
$$\delta(\ell_2) = -\delta(\ell_1),$$
and since $2H$ is rational as well,
$$\delta(2H - \ell_1) \;=\; -\delta(\ell_1) \;=\; \delta(\ell_2).$$
So the obstruction met in §II.6 is the class of the *other* blow-down: the same as one of the two
Severi–Brauer surfaces and the inverse of the other. That is why "same class or its inverse" is the
right shape of answer — there genuinely are two, and they are opposite.

In [20]:
def blowdown_class(sixer):
    "the unique D with D.L = 0 for L in the sixer and D.K = -3"
    basis = identity_matrix(ZZ, 7).rows()
    Mrow = Matrix(ZZ, [[dot(tuple(b), Lq) for b in basis] for Lq in sixer]
                    + [[dot(tuple(b), Kc) for b in basis]])
    return tuple(Mrow.solve_right(vector(ZZ, [0]*len(sixer) + [-3])))

six1 = list(Tri) + list(tri2[0])
six2 = list(Tri) + list(tri2[1])
l1, l2 = blowdown_class(six1), blowdown_class(six2)
Mtri = tuple(sum(vector(ZZ, t) for t in Tri))
print("l1 =", l1, "  (D^2, D.K) =", (dot(l1,l1), dot(l1,Kc)))
print("l2 =", l2, "  (D^2, D.K) =", (dot(l2,l2), dot(l2,Kc)))
print()
print("l1 + l2 =", tuple(vector(ZZ,l1)+vector(ZZ,l2)))
print("H  + M  =", tuple(vector(ZZ,Hc)+vector(ZZ,Mtri)),
      "   equal ?", tuple(vector(ZZ,l1)+vector(ZZ,l2)) == tuple(vector(ZZ,Hc)+vector(ZZ,Mtri)))

l1 = (1, 0, 0, 0, 0, 0, 0)   (D^2, D.K) = (1, -3)
l2 = (2, 0, 0, 0, -1, -1, -1)   (D^2, D.K) = (1, -3)

l1 + l2 = (3, 0, 0, 0, -1, -1, -1)
H  + M  = (3, 0, 0, 0, -1, -1, -1)    equal ? True


In [21]:
d2 = tuple(2*vector(ZZ,Hc) - vector(ZZ,l1))
print("2H - l1 =", d2, "   (D^2, D.K) =", (dot(d2,d2), dot(d2,Kc)), " -> also a blow-down class")
sx = [Lq for Lq in lines if dot(d2, Lq) == 0]
print("   its sixer: ", len(sx), "lines, pairwise skew ?",
      all(dot(a,b) == 0 for a, b in combinations(sx, 2)))
print("   equals l2 ?", d2 == l2, "  -- a THIRD sixer, but carrying the same Brauer class as l2")
print()
print("all 72 blow-down classes have (D^2, D.K) = (1,-3); the three above are three of them:")
print("   ", [(dot(c,c), dot(c,Kc)) for c in (l1, l2, d2)])

2H - l1 = (5, -2, -2, -2, -2, -2, -2)    (D^2, D.K) = (1, -3)  -> also a blow-down class
   its sixer:  6 lines, pairwise skew ? True
   equals l2 ? False   -- a THIRD sixer, but carrying the same Brauer class as l2

all 72 blow-down classes have (D^2, D.K) = (1,-3); the three above are three of them:
    [(1, -3), (1, -3), (1, -3)]


**Consequence: the recipe is sharp.** In the stable-sixer case,
$$S \text{ is } k\text{-rational} \iff \delta(\ell_1) = 0 \iff S(k) \neq \emptyset,$$
the middle implication because a rational point splits $\operatorname{Pic}(S) \to \operatorname{Pic}(\bar S)^G$. Since a
rational $\Gamma \in |2H-\ell|$ exists exactly when $\delta(2H-\ell) = -\delta(\ell_1)$ vanishes, the net
of quadrics through $\Gamma$ can be written over $k$ **precisely when a birational map to $\mathbb{P}^2$
exists at all**. The method's failure is therefore a correct verdict, not a limitation of the method.

In the worked example there is no obstruction, because the surface visibly has rational points — it was
built as the image of $\mathbb{P}^2$:

In [22]:
p = tuple(gg(1, 0, 0) for gg in f)
print("image of (1:0:0) in P^3:", p, " ~ ", tuple(QQ(c)/2 for c in p))
print("lies on S ?", Fs(*p) == 0)
print("so delta vanishes on all invariant classes, and the rational net of II.6 had to exist.")

image of (1:0:0) in P^3: (2, 0, 0, 4)  ~  (1, 0, 0, 2)
lies on S ? True
so delta vanishes on all invariant classes, and the rational net of II.6 had to exist.


## Summary

**The mechanism.** $L(D) \cong \{\varphi \text{ of degree } m : \operatorname{div}\varphi \ge \operatorname{div}(h) - D\}$
via $f \mapsto fh$. Choose a denominator, impose vanishing, do linear algebra. You need $D$ as a
divisor, not a class, and you need the multiplicities right.

**Curves.** On a smooth plane model both requirements are trivial and $L(mH-A)$ is just "forms of
degree $m$ through $A$"; the canonical system is the forms of degree $d-3$. On a singular model the
multiplicities are *not* trivial — the naive count gave $3$ where the truth was $5$ — and the classical
repair is the conductor/adjoint bookkeeping of Brill–Noether. Magma and Sage instead use Hess's
algorithm in orders of the function field, which sidesteps plane models entirely. The adjoint systems
are not merely a dimension count: the conics through the three nodes parametrised the quartic outright.

**Cubic surfaces.** The problem splits very unevenly.

- Contracting the Galois-stable triple of skew lines is **free**: the three projections
  $\pi_i = (h_1^{(i)} : h_2^{(i)})$ assemble into $S \dashrightarrow R_{F/\mathbb{Q}}\mathbb{P}^1_F$, a birational
  map onto a del Pezzo surface of degree $6$. Nothing but linear forms is needed.
- Getting to $\mathbb{P}^2$ needs a Galois-stable **sixer**, which exists iff Galois preserves (rather than
  swaps) the two triples of lines skew to all three. Then $\ell = H + E - L^\vee$ and
  $$L(\ell) \;=\; \tfrac{1}{q_\Gamma}\{\text{quadrics containing } \Gamma\}, \qquad \Gamma \in |2H-\ell|
  \text{ a twisted cubic on } S,$$
  a net, because a twisted cubic imposes $7$ conditions on the $10$ quadrics.
- Over the field of the lines, $\Gamma = C' + L^\vee$ (conic $+$ line) is free, and the map is written
  down and verified birational.
- Over $\mathbb{Q}$ one needs a **rational** $\Gamma$. No change of basis achieves this — the denominator
  itself is irrational — and the obstruction is the Severi–Brauer surface $|2H-\ell|$, split as soon as
  $S(\mathbb{Q}) \neq \emptyset$. That obstruction is not an artefact: contracting the two triples gives two
  Severi–Brauer surfaces with **opposite** classes, $\delta(\ell_2) = -\delta(\ell_1)$ (from
  $\ell_1+\ell_2 = H+M$), and $\delta(2H-\ell_1) = \delta(\ell_2)$; when the class is non-trivial $S$ is
  not $k$-rational at all, so the recipe is sharp rather than lossy (§II.7). In this example a rational
  $\Gamma$ is produced from the quintics with six double points, giving
  $$(Y_0:Y_1:Y_2:Y_3) \;\longmapsto\; \big(4Y_0^2 - 3Y_1Y_2 - 2Y_0Y_3 \,:\, 10Y_0Y_1 - 2Y_2^2 - 5Y_1Y_3
  \,:\, 4Y_0Y_2 - 15Y_1^2\big).$$

So the honest shape of the answer to the original question is: *Riemann–Roch spaces are linear algebra
once you have an explicit divisor; on a cubic surface the geometry hands you the divisor over the
field of the lines, and what is left is not a harder computation but a genuine arithmetic invariant —
a class in $\operatorname{Br}(k)[3]$ whose vanishing is equivalent to the map existing.*